# Notebook 05: Stochastic Sheet Forcing and Fragmentation Statistics

This notebook upgrades the repo from **one deterministic trajectory** to an **ensemble of randomized runs**.

Notebook 04 asked:

- how does one fragmented current sheet evolve in pseudo-time?

Notebook 05 asks:

- are the fragmented pocket statistics robust across randomized sheet-localized forcing?
- do width variation, pocket count, and 45°-constraint violation remain bounded across runs?

This notebook keeps the same core gate

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

and studies an ensemble of pseudo-time evolutions under randomized seed phases, amplitude jitter, and optional weak stochastic forcing near the sheet.

This is still **not** a full plasma solver. It is an ensemble geometric-dynamics notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 360, 280
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Shared helpers

We reuse the same geometry and pseudo-time update structure as Notebook 04, but now randomize the forcing.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def initial_field(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = frag_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))

    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By, ks, phases, amps

def random_seed_pattern(X, Y, base_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = base_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))
    return multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def activity_map(cos_map, threshold=THRESHOLD):
    act = np.maximum(0.0, threshold - cos_map)
    return np.nan_to_num(act, nan=0.0)

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0
    sizes = []

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True
                size = 0

                while q:
                    r, c = q.popleft()
                    size += 1
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
                sizes.append(size)
    return count, sizes

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def local_min_count_1d(arr):
    arr = np.asarray(arr, dtype=float)
    good = np.isfinite(arr)
    idx = np.where(good)[0]
    if len(idx) < 3:
        return 0
    vals = arr[idx]
    count = 0
    for k in range(1, len(vals) - 1):
        if vals[k] < vals[k - 1] and vals[k] < vals[k + 1]:
            count += 1
    return count

def constraint_energy(cos_map, threshold=THRESHOLD):
    return float(np.nanmean(np.maximum(0.0, threshold - cos_map)))

def one_step(Bx, By, Bx_target, seed_pattern, rng, dt, dx, dy, nu, alpha, beta, gamma, noise_amp=0.0, noise_sigma=0.38):
    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    act = activity_map(cos_map, threshold=THRESHOLD)

    sheet_noise = noise_amp * rng.standard_normal(X.shape) * np.exp(-(Y**2) / (noise_sigma**2))

    Bx_new = Bx + dt * (
        nu * laplacian(Bx, dx, dy)
        - alpha * (Bx - Bx_target)
    )

    By_new = By + dt * (
        nu * laplacian(By, dx, dy)
        + beta * act * seed_pattern
        - gamma * By
        + sheet_noise
    )

    Bx_new[0, :] = Bx_target[0, :]
    Bx_new[-1, :] = Bx_target[-1, :]
    By_new[0, :] = 0.0
    By_new[-1, :] = 0.0

    return Bx_new, By_new

## 2. Ensemble setup

We define one baseline parameter set and then run many randomized realizations.

In [ ]:
delta = 0.35
frag_amp = 0.28
sigma_seed = 0.38

params = {
    "dt": 0.0018,
    "nu": 0.0012,
    "alpha": 0.25,
    "beta": 0.85,
    "gamma": 0.18,
    "noise_amp": 0.004,
    "noise_sigma": 0.38,
    "n_steps": 180,
}

n_runs = 24
save_steps = [0, 60, 120, 180]

params

## 3. One-run simulator

This function returns both time-series metrics and final-state summaries for one randomized run.

In [ ]:
def run_one_ensemble_member(run_id, params):
    rng = np.random.default_rng(1000 + run_id)

    Bx0, By0, ks, phases, amps = initial_field(
        X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=sigma_seed, rng=rng
    )
    Bx_target = harris_target(X, Y, delta=delta)
    seed_pattern = random_seed_pattern(X, Y, base_amp=frag_amp, sigma=sigma_seed, rng=rng)

    Bx_t = Bx0.copy()
    By_t = By0.copy()

    metrics = {
        "step": [],
        "failed_fraction": [],
        "central_failed_fraction": [],
        "component_count": [],
        "width_std": [],
        "centerline_min": [],
        "constraint_energy": [],
    }

    snapshots = {}

    for step in range(params["n_steps"] + 1):
        Bx_hat_t, By_hat_t = normalize_field(Bx_t, By_t)
        cos_map_t = cross_sheet_cosine(Bx_hat_t, By_hat_t, shift=3)
        mask_t = failed_mask(cos_map_t)
        central_t = central_band_mask(mask_t, Y, half_width=0.8)

        comp_count, comp_sizes = connected_components(np.nan_to_num(central_t, nan=False).astype(bool))
        widths_t = estimate_failed_width_vs_x(cos_map_t, y, threshold=THRESHOLD)

        center_row = np.argmin(np.abs(y - 0.0))
        centerline_t = cos_map_t[center_row, :]

        metrics["step"].append(step)
        metrics["failed_fraction"].append(float(np.nanmean(mask_t)))
        metrics["central_failed_fraction"].append(float(np.nanmean(central_t)))
        metrics["component_count"].append(int(comp_count))
        metrics["width_std"].append(float(np.nanstd(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
        metrics["centerline_min"].append(float(np.nanmin(centerline_t)))
        metrics["constraint_energy"].append(constraint_energy(cos_map_t))

        if step in save_steps:
            snapshots[step] = {
                "cos_map": cos_map_t.copy(),
                "mask": mask_t.copy(),
                "central": central_t.copy(),
            }

        if step < params["n_steps"]:
            Bx_t, By_t = one_step(
                Bx_t, By_t, Bx_target, seed_pattern, rng,
                dt=params["dt"], dx=dx, dy=dy,
                nu=params["nu"], alpha=params["alpha"], beta=params["beta"], gamma=params["gamma"],
                noise_amp=params["noise_amp"], noise_sigma=params["noise_sigma"]
            )

    crit_step = int(np.argmax(np.gradient(metrics["failed_fraction"])))

    final_summary = {
        "run_id": run_id,
        "final_failed_fraction": metrics["failed_fraction"][-1],
        "final_central_failed_fraction": metrics["central_failed_fraction"][-1],
        "final_component_count": metrics["component_count"][-1],
        "final_width_std": metrics["width_std"][-1],
        "final_centerline_min": metrics["centerline_min"][-1],
        "final_constraint_energy": metrics["constraint_energy"][-1],
        "critical_step_proxy": crit_step,
    }

    return metrics, snapshots, final_summary

## 4. Run the ensemble

In [ ]:
ensemble_metrics = []
ensemble_summaries = []
example_snapshots = {}

for run_id in range(n_runs):
    metrics_i, snapshots_i, summary_i = run_one_ensemble_member(run_id, params)
    ensemble_metrics.append(metrics_i)
    ensemble_summaries.append(summary_i)

    if run_id in [0, 1, 2]:
        example_snapshots[run_id] = snapshots_i

print("completed runs =", len(ensemble_summaries))
ensemble_summaries[:3]

## 5. Example snapshot panels from a few runs

This helps show that the ensemble varies in detail while preserving the same qualitative structure.

In [ ]:
for run_id in [0, 1, 2]:
    for step in save_steps:
        snap = example_snapshots[run_id][step]
        plt.figure(figsize=(9, 4.6))
        plt.imshow(
            snap["cos_map"],
            extent=[x.min(), x.max(), y.min(), y.max()],
            origin="lower",
            aspect="auto"
        )
        plt.colorbar(label="cross-sheet cosine")
        plt.contour(
            X, Y, snap["cos_map"],
            levels=[THRESHOLD],
            colors="white",
            linewidths=1.8,
            linestyles="--"
        )
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title(f"Run {run_id} Cross-Sheet Cosine Map (step={step})")
        plt.show()

In [ ]:
for run_id in [0, 1, 2]:
    for step in save_steps:
        snap = example_snapshots[run_id][step]
        plt.figure(figsize=(9, 4.6))
        plt.imshow(
            snap["central"],
            extent=[x.min(), x.max(), y.min(), y.max()],
            origin="lower",
            aspect="auto"
        )
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title(f"Run {run_id} Central Failed Region (step={step})")
        plt.show()

## 6. Collect summary arrays

In [ ]:
final_component_count = np.array([s["final_component_count"] for s in ensemble_summaries])
final_width_std = np.array([s["final_width_std"] for s in ensemble_summaries])
final_constraint_energy = np.array([s["final_constraint_energy"] for s in ensemble_summaries])
final_failed_fraction = np.array([s["final_failed_fraction"] for s in ensemble_summaries])
critical_step_proxy = np.array([s["critical_step_proxy"] for s in ensemble_summaries])

print("mean final pocket count =", np.mean(final_component_count))
print("mean final width std =", np.mean(final_width_std))
print("mean final constraint energy =", np.mean(final_constraint_energy))

## 7. Final-state histograms

These are the main ensemble distributions.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(final_component_count, bins=np.arange(final_component_count.min(), final_component_count.max() + 2) - 0.5)
plt.xlabel("final pocket count")
plt.ylabel("count")
plt.title("Final Pocket Count Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(final_width_std, bins=12)
plt.xlabel("final width std")
plt.ylabel("count")
plt.title("Final Width-Variation Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(final_constraint_energy, bins=12)
plt.xlabel("final constraint energy")
plt.ylabel("count")
plt.title("Final Constraint-Energy Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(critical_step_proxy, bins=12)
plt.xlabel("critical-step proxy")
plt.ylabel("count")
plt.title("Critical-Step Proxy Distribution")
plt.show()

## 8. Scatter plots

These show relationships among the final ensemble observables.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(final_constraint_energy, final_component_count)
plt.xlabel("final constraint energy")
plt.ylabel("final pocket count")
plt.title("Pocket Count vs Constraint Energy")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(final_width_std, final_constraint_energy)
plt.xlabel("final width std")
plt.ylabel("final constraint energy")
plt.title("Width Variation vs Constraint Energy")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(critical_step_proxy, final_constraint_energy)
plt.xlabel("critical-step proxy")
plt.ylabel("final constraint energy")
plt.title("Critical Step vs Constraint Energy")
plt.show()

## 9. Ensemble mean ± std time series

This is the cleanest summary of how the whole ensemble behaves over pseudo-time.

In [ ]:
steps = np.array(ensemble_metrics[0]["step"])

failed_matrix = np.array([m["failed_fraction"] for m in ensemble_metrics])
width_std_matrix = np.array([m["width_std"] for m in ensemble_metrics])
energy_matrix = np.array([m["constraint_energy"] for m in ensemble_metrics])

failed_mean = np.mean(failed_matrix, axis=0)
failed_std = np.std(failed_matrix, axis=0)

width_mean = np.mean(width_std_matrix, axis=0)
width_std = np.std(width_std_matrix, axis=0)

energy_mean = np.mean(energy_matrix, axis=0)
energy_std = np.std(energy_matrix, axis=0)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, failed_mean)
plt.fill_between(steps, failed_mean - failed_std, failed_mean + failed_std, alpha=0.25)
plt.xlabel("step")
plt.ylabel("below-threshold fraction")
plt.title("Ensemble Mean ± Std: Below-Threshold Fraction")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, width_mean)
plt.fill_between(steps, width_mean - width_std, width_mean + width_std, alpha=0.25)
plt.xlabel("step")
plt.ylabel("width std along x")
plt.title("Ensemble Mean ± Std: Width Variation")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, energy_mean)
plt.fill_between(steps, energy_mean - energy_std, energy_mean + energy_std, alpha=0.25)
plt.xlabel("step")
plt.ylabel("constraint energy")
plt.title("Ensemble Mean ± Std: Constraint Energy")
plt.show()

## 10. Compact run summary table

This prints one line per ensemble member.

In [ ]:
for s in ensemble_summaries:
    print(
        f"run={s['run_id']:02d} | "
        f"final_pockets={s['final_component_count']} | "
        f"final_width_std={s['final_width_std']:.4f} | "
        f"final_constraint_energy={s['final_constraint_energy']:.4f} | "
        f"critical_step={s['critical_step_proxy']}"
    )

## 11. Interpretation

Notebook 05 upgrades the repo from a deterministic example to a statistical statement.

What it supports:

> Under randomized sheet-localized forcing, the 45° gate organizes a bounded ensemble of fragmented current-sheet states.

What makes this stronger than Notebook 04:

- many randomized realizations are tested
- final pocket count, width variation, and constraint energy become distributions
- ensemble mean ± std curves show whether the behavior is robust or seed-dependent

What it still does **not** claim:

- full reconnection transport physics
- solar-flare prediction
- a substitute for MHD or PIC simulation

Best interpretation:

> an ensemble geometric precursor model for bounded current-sheet fragmentation under stochastic sheet forcing.